# Getting Started with MLflow for Traditional Machine Learning

A beginner-friendly, hands-on tour of MLflow using a real climate dataset: the NOAA Global Historical Climatology Network daily observations, available in this workspace through the Databricks Marketplace. We will train regression models that predict a day's maximum temperature, and use MLflow to track, compare, and package every experiment along the way.

## 1. What is MLflow??

**MLflow** is an open-source platform for managing the end-to-end machine learning lifecycle. Instead of losing track of your experiments in scattered notebooks and spreadsheets, MLflow gives you a systematic way to record, organize, compare, and deploy your work.

MLflow has a few core pieces. In this notebook we focus on the first two, and point to the third at the end:

| Component | What it does | Where you see it |
| --- | --- | --- |
| **MLflow Tracking** | Records every training run: parameters, metrics, models, artifacts | The Experiments page in the workspace |
| **Model Logging** | Packages a trained model (with its preprocessing and signature) so it can be reused or deployed | The model artifact inside each run |
| **Model Registry** | Gives packaged models a name and version in Unity Catalog, ready to serve | Catalog → your schema → Models |

**The workflow in this notebook:** load data → explore → train with *manual* logging → train with *autologging* → compare runs → log a deployable model. Each section builds on the previous one.


## 2. Our Dataset: Daily Climate Observations (NOAA GHCN-Daily)

We will use the **NOAA Global Historical Climatology Network - Daily** dataset, available in this workspace through the Databricks Marketplace. Each row is one weather station's observation for one day: its location (latitude, elevation) and its daily measurements (maximum and minimum temperature).

Our regression question: **can we predict a day's maximum temperature from the day's minimum temperature, the station's location, and the time of year?**

The table stores GHCN's raw units (temperatures in tenths of a degree Celsius), so converting to familiar units is part of loading the data.

In [0]:
# Load one year of daily observations for US stations and convert to familiar units
import pandas as pd

query = """
    SELECT station, name, date, latitude, elevation,
           temp_max, temp_min
    FROM rearc_daily_weather_observations_noaa.esg_noaa_ghcn.noaa_ghcn_daily
    WHERE station LIKE 'US%'
      AND date BETWEEN '2023-01-01' AND '2023-12-31'
      AND temp_max IS NOT NULL
      AND temp_min IS NOT NULL
"""

climate_df = spark.sql(query).toPandas()

# GHCN stores temperatures in tenths of a degree Celsius: 256 means 25.6 C.
# Convert to degrees Celsius so the numbers we work with are human-friendly.
climate_df["date"] = pd.to_datetime(climate_df["date"])
climate_df["temp_max_c"] = climate_df["temp_max"] / 10.0
climate_df["temp_min_c"] = climate_df["temp_min"] / 10.0
climate_df["month"] = climate_df["date"].dt.month

print(f"Loaded {len(climate_df):,} daily observations")
display(climate_df.head(10))

In [0]:
# A quick look before modeling: size, target distribution, and how features relate to it
import matplotlib.pyplot as plt

print(f"Rows: {climate_df.shape[0]:,}   Stations: {climate_df['station'].nunique():,}")
display(climate_df[["temp_max_c", "temp_min_c", "latitude", "elevation", "month"]].describe().round(1))

corr = climate_df[["temp_min_c", "latitude", "elevation", "month", "temp_max_c"]].corr()["temp_max_c"].sort_values(ascending=False)
print("Correlation with the target (temp_max_c):")
print(corr.round(3))

# Visualize the strongest relationship - the one our regression will lean on
sample = climate_df.sample(n=20_000, random_state=42)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sample["temp_min_c"], sample["temp_max_c"], s=4, alpha=0.2, color="tab:blue")
ax.set_xlabel("Daily minimum temperature (C)")
ax.set_ylabel("Daily maximum temperature (C)")
ax.set_title("Daily low vs daily high - the relationship our regression will learn")
plt.show()

## 3. MLflow Tracking: Experiments, Runs, Parameters, Metrics

Four words explain 90% of MLflow:

* **Experiment** - a named folder that groups related runs. Ours will be `mlflow_basics_climate_regression`.
* **Run** - one execution of training. Every time you fit a model inside `mlflow.start_run()`, that is a run with a unique `run_id`.
* **Parameters** - the inputs you chose: model type, hyperparameters, feature list, test split size.
* **Metrics** - the numbers that tell you how good the model is. For regression: RMSE, MAE, R-squared.

Because parameters and metrics live together in each run, MLflow can answer the question we normally answer with messy spreadsheets: *which settings produced the best model?*

On Databricks, experiment names must be absolute workspace paths, e.g. `/Users/<your-email>/mlflow_basics_climate_regression`.

In [0]:
# --- Manual logging: we decide exactly what gets recorded ---
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

EXPERIMENT_PATH = "/Users/doctornikhildjonathan@gmail.com/mlflow_basics_climate_regression"
mlflow.set_experiment(EXPERIMENT_PATH)

feature_cols = ["temp_min_c", "latitude", "elevation", "month"]
X = climate_df[feature_cols]
y = climate_df["temp_max_c"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run(run_name="linear_regression_manual") as run:
    model = LinearRegression()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)

    # Everything inside the `with` block is recorded to this run
    mlflow.log_params({
        "model_type": "LinearRegression",
        "features": ",".join(feature_cols),
        "test_size": 0.2,
        "random_state": 42,
    })
    mlflow.log_metrics({
        "mae_c": mae,
        "rmse_c": rmse,
        "r2": r2,
    })

    print(f"Run id: {run.info.run_id}")
    print(f"MAE: {mae:.2f} C | RMSE: {rmse:.2f} C | R2: {r2:.3f}")

## 4. Autologging: Let MLflow Record Everything For You

Manual logging is explicit and precise, but it is easy to forget a metric. With `mlflow.sklearn.autolog()`, MLflow instruments scikit-learn itself: it automatically records the estimator's hyperparameters, training metrics, and (if enabled) the fitted model - no `log_params` or `log_metrics` calls needed.

We will train a **Ridge** regression next. Ridge is linear regression plus L2 regularization - one extra hyperparameter, `alpha`, that shrinks coefficients to reduce overfitting. Autolog will capture `alpha` for us automatically.

In [0]:
from sklearn.linear_model import Ridge

# Turn on autologging - from here on, MLflow records sklearn fits automatically.
# We disable model logging here to keep the focus on params/metrics (we log a
# deployable model explicitly in section 6).
mlflow.sklearn.autolog(log_models=False)

with mlflow.start_run(run_name="ridge_autolog") as run:
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, y_train)

    ridge_predictions = ridge.predict(X_test)
    # Even with autolog on, you can still add your own metrics on top
    mlflow.log_metric("rmse_c", mean_squared_error(y_test, ridge_predictions) ** 0.5)

print("This run was recorded with zero manual log_params / log_metrics calls.")
print("Check the Experiments sidebar: alpha and other settings were captured automatically.")

## 5. Comparing Runs: Your MLflow Leaderboard

Individual runs are useful; comparing runs is where MLflow pays off. `MlflowClient.search_runs` reads every run in the experiment, so you can build a leaderboard with a few lines of code. This is the same data you see in the Experiments UI - here sorted by RMSE (lower is better).

In [0]:
# --- Compare every run in the experiment: your MLflow leaderboard ---
import pandas as pd
from mlflow import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_PATH)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse_c ASC"],
)

leaderboard = pd.DataFrame([
    {
        "run_name": r.data.tags.get("mlflow.runName"),
        "rmse_c": r.data.metrics.get("rmse_c"),
        "mae_c": r.data.metrics.get("mae_c"),
        "r2": r.data.metrics.get("r2"),
        "alpha": r.data.params.get("alpha"),
        "run_id": r.info.run_id,
    }
    for r in runs
])
display(leaderboard)

## 6. Logging a Deployable Model: Pipelines and Signatures

Metrics tell you which model is best; `log_model` packages the winner so others can reuse it. Two things make a logged model production-ready:

* **A pipeline, not a bare model.** We bundle scaling + regression into one sklearn `Pipeline`, so the model carries its own preprocessing - whoever loads it sends raw features, exactly as during training.
* **A signature and an input example.** The signature declares the expected input columns and output; the input example shows a caller what a real request looks like.

We log the Ridge model as the final deployable artifact of a fresh run.

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from mlflow.models import infer_signature

# We log the final model explicitly, so turn autologging off first
mlflow.sklearn.autolog(disable=True)

with mlflow.start_run(run_name="final_ridge_pipeline") as run:
    final_model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
    ])
    final_model.fit(X_train, y_train)

    final_predictions = final_model.predict(X_test)
    mlflow.log_params({"model_type": "Ridge", "alpha": 1.0, "features": ",".join(feature_cols)})
    mlflow.log_metrics({
        "rmse_c": mean_squared_error(y_test, final_predictions) ** 0.5,
        "mae_c": mean_absolute_error(y_test, final_predictions),
        "r2": r2_score(y_test, final_predictions),
    })

    # Signature: what the model expects as input and produces as output
    signature_input = X_train.head(100)
    signature = infer_signature(signature_input, final_model.predict(signature_input))

    model_info = mlflow.sklearn.log_model(
        final_model,
        name="climate_temp_regressor",
        signature=signature,
        input_example=X_train.head(3),
    )
    print(f"Logged model URI: {model_info.model_uri}")
    print(f"Run id: {run.info.run_id}")

# Proof the model travels with its preprocessing: reload and predict from raw features
import mlflow.pyfunc
reloaded = mlflow.pyfunc.load_model(model_info.model_uri)
print("Reloaded model prediction (C):", reloaded.predict(X_test.head(3)).round(1).tolist())

## 7. Summary and Where to Go Next

**What you did in this notebook:**

1. Loaded a real marketplace climate dataset and framed a regression question
2. Learned MLflow's vocabulary: experiment → run → parameters → metrics
3. Logged a run manually (full control) and with autologging (zero effort)
4. Compared all runs programmatically to find the best model
5. Packaged the winner as a deployable model with a signature and input example

**Where to go next:**

* **Register the model in Unity Catalog** - give `climate_temp_regressor` a three-part name and a version under Catalog → Models, so it can be shared, governed, and deployed
* **Serve it** - deploy a registered model to a Model Serving endpoint as a REST API
* **Experiment more** - try other model families (random forest, gradient boosting) in the same experiment; each new `mlflow.start_run` adds a row to your leaderboard